# التقييم النهائي للأنظمة الثلاثة على Test

شغّل خلية الكود الوحيدة بعد تفعيل GPU وإيقاف أي تدريب. الدفتر مستقل عن متغيرات الجلسة السابقة، ولا يحتاج إلى feature_cache أو Internet.

## الملفات اللازمة

- الأوزان الكاملة المختارة وإيصال SHA256 المجاور:
`/kaggle/working/qr_three_systems_random_fusion_v1/selected_three_systems_epoch_01_b06b88974bbe.pt`
- مجلد `tokenizer` في مجلد التجربة نفسه، كما في حزمة الأوزان التي نزّلتها.
- صور QR الأصلية في Input، وCSV الأصلي، والقائمة المدققة وaudit.json في المسارات السابقة.

## الأنظمة التي نقارنها

1. **visual:** MobileNet البصري بعد تحسين المتانة.
2. **text:** BERT-Tiny المحفوظ، على الروابط المستخرجة مسبقًا في CSV.
3. **feature_fusion:** رأس الدمج المتعلّم المختار من الحقبة1؛ يستقبل256ميزة بصرية و256نصية. عند غياب الرابط يرجع النظام إلى البصري، ويجب ذكر هذه السياسة عند وصفه.
4. **text_with_visual_fallback:** خط أساس إضافي يستخدم النصي عند توفر الرابط والبصري عند غيابه، دون رأس الدمج المتعلّم.

تُقارن الأنظمة على الأفراد أنفسهم، وتُفصل النتائج إلى all وurl_available وurl_missing. الرابط متاح عندما يكون ok=1 ونصه غير فارغ. نتيجة النصي عند غياب الرابط تعتمد على المدخل الافتراضي المحفوظ، وليست كشفًا دلاليًا لرابط تعذّر استخراجه.

## المنهجية والمخرجات

الأوزان مثبتة عند الحقبة المختارة، والعتبة0.5، والاستدلالFP32 مع التحويلات الأصلية. لا تدريب أو بحث عتبات أو اختيار نموذج باستخدامTest. Test سبق فتحه لتجارب البصري، وتُوثّق إعادة استخدامه. أوزان النصي الأصلية لا تحمل بصمة تقسيم التدريب؛ احتفظ بتوثيق عضوية Train القديمة قبل الادعاء باختبار مستقل.

المقاييس: Accuracy وBalancedAccuracy وMacro-F1 وPrecision/Recall/F1 للخبيث وROC-AUC وAveragePrecision وأعدادTN/FP/FN/TP. Macro-F1 يحسب الفئتين صراحةً حتى في المجموعات الفرعية، وAUC غير معرّف للمجموعة ذات فئة فعلية واحدة. تُحفظ تقارير الفئات ومصفوفات الالتباس ومنحنياتROC وPrecision–Recall، والتوقعات مرتبة مع مسارات الصور ومجموعاتها. توجد أعداد أخطاء مقترنة للمقارنة الوصفية، دون ادعاء دلالة إحصائية تفترض استقلال الصور.

الروابط مصدرها القائمة المحفوظة؛ لا يُعاد فك QR ولا تُفتح الروابط. لا نقيس زمن الخط الكامل من التصوير إلى فك الرمز. هذه النتائج لا تكفي وحدها لتقييم صور مطبوعة أو مصوّرة بالجوال أو تشويهات جديدة.

## التشغيل

يتحقق الكود من SHA256 والـtokenizer والتقسيم، ويحمّل جميع الطبقات بـstrict=True دون تنزيل أوزان. يقرأ146425صورةTest مرة واحدة مع شريط تقدم ورسالة كل100دفعة. عند خطأ عمال التحميل يمكن تغيير WORKERSإلى0 وإعادة الخلية؛ أبقِ العتبة والتحويلات والأوزان ثابتة.

كل تشغيل ينشئ مجلد نتائج جديدًا وحزمةZIP، دون تغيير الأوزان. الوقت يعتمد على سرعة التخزين وتجهيز الصور وGPU؛ راقب شريط التقدم.

بعد اكتماله أرسل:
`test_all_systems_summary.csv`
`test_three_systems_comparison.csv`

نزّل ZIPالنتائج إلى جهازك. فُحصت الصياغة والمقاييس التجريبية وسياسة الرجوع محليًا؛ لم يُشغّل النموذج على GPU أو تُقرأ أوزانك الفعلية في بيئة المساعد.


In [ ]:
print('FINAL TEST: visual, text, learned feature fusion + routing baseline. No training.',flush=True)
import os,json,time,uuid,hashlib,zipfile,gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms as T
from PIL import Image
from tqdm import tqdm
from transformers import BertConfig,BertModel,AutoTokenizer
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,f1_score,precision_score,recall_score,
    roc_auc_score,average_precision_score,confusion_matrix,classification_report,ConfusionMatrixDisplay,
    roc_curve,precision_recall_curve)
import matplotlib.pyplot as plt

RUN=Path('/kaggle/working/qr_three_systems_random_fusion_v1')
WEIGHTS=RUN/'selected_three_systems_epoch_01_b06b88974bbe.pt'
TOKENIZER_DIR=RUN/'tokenizer'
ROOT=Path('/kaggle/working/qr_visual_fresh')
SOURCE_CSV=Path('/kaggle/input/datasets/feras2004/spliteoriginal/splitsOriginal.csv')
EXPECTED_SIGNATURE='d4509181f834a16144ad45e24b8ce686a4e608e07a9846a880757c705739d1ff'
THRESHOLD=0.5
BATCH_SIZE=128
WORKERS=2
MAX_LEN=128
SYSTEMS=['visual','text','feature_fusion','text_with_visual_fallback']
assert torch.cuda.is_available(),'Enable GPU first. CUDA must be True.'
DEVICE=torch.device('cuda:0');torch.set_num_threads(1)
os.environ['TOKENIZERS_PARALLELISM']='false'
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
print('GPU:',torch.cuda.get_device_name(0),flush=True)


def file_hash(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()


def atomic_json(value,path):
    path=Path(path);tmp=path.with_name(path.name+'.writing')
    with tmp.open('w',encoding='utf-8') as f:
        json.dump(value,f,ensure_ascii=False,indent=2,allow_nan=False);f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)


def data_signature(frame):
    h=hashlib.sha256()
    for row in tqdm(frame[['path','label','split','url','ok','sha256']].itertuples(index=False,name=None),
                    total=len(frame),desc='Verify data signature',mininterval=2):
        h.update((json.dumps(row,ensure_ascii=False,separators=(',',':'))+'\n').encode())
    return h.hexdigest()

print('[1/4] Verify the fixed trained checkpoint and tokenizer...',flush=True)
assert WEIGHTS.is_file(),f'Missing checkpoint: {WEIGHTS}'
receipt=json.loads(WEIGHTS.with_suffix('.json').read_text())
assert receipt['file']==WEIGHTS.name
WEIGHTS_SHA=file_hash(WEIGHTS);assert WEIGHTS_SHA==receipt['sha256']
with zipfile.ZipFile(WEIGHTS) as archive:assert archive.testzip() is None
checkpoint=torch.load(WEIGHTS,map_location='cpu',weights_only=False) # your own trusted file only
assert checkpoint['epoch']==1,'This evaluation is fixed to the selected trained epoch1.'
assert checkpoint['identity']['data_signature']==EXPECTED_SIGNATURE
assert checkpoint['cfg']['initialization']=='random'
assert checkpoint['cfg']['threshold']==THRESHOLD
assert checkpoint['missing_url_policy']=='visual fallback; feature head used only when URL is available'
source_identity=checkpoint['source_model_identity']
assert source_identity['data_signature']==EXPECTED_SIGNATURE
assert source_identity['cfg']['max_len']==MAX_LEN
assert checkpoint['identity']['visual_sha256']==source_identity['visual_sha256']
assert checkpoint['identity']['text_sha256']==source_identity['text_sha256']
state=checkpoint['model_state']
for key,value in state.items():
    assert torch.is_tensor(value),key
    if value.is_floating_point():assert torch.isfinite(value).all(),key
assert TOKENIZER_DIR.is_dir(),'Restore the tokenizer folder included in the export ZIP.'
for name,sha in source_identity.get('tokenizer_files_sha256',{}).items():
    assert file_hash(TOKENIZER_DIR/name)==sha,f'Tokenizer file differs: {name}'
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),use_fast=True,local_files_only=True)
assert len(tokenizer)==30522 and tokenizer.cls_token_id==101 and tokenizer.sep_token_id==102 and tokenizer.pad_token_id==0
vocab_hash=hashlib.sha256(json.dumps(tokenizer.get_vocab(),sort_keys=True,separators=(',',':')).encode()).hexdigest()
assert vocab_hash==source_identity['vocab_sha256']

print('[2/4] Verify original Test membership...',flush=True)
audit=json.loads((ROOT/'audit.json').read_text())
assert audit['data_signature']==EXPECTED_SIGNATURE and audit.get('train_ready') is True
frame=None
for path in [ROOT/'verified_mobilenet_inputs/splits_verified.parquet',
             ROOT/'verified_mobilenet_inputs/splits_verified.csv.gz',ROOT/'splits_clean.parquet']:
    if not path.is_file():continue
    try:
        candidate=pd.read_parquet(path) if path.suffix=='.parquet' else pd.read_csv(path,keep_default_na=False)
        assert len(candidate)==1005700 and candidate.path.is_unique
        assert candidate.label.isin([0,1]).all() and candidate.ok.isin([0,1]).all()
        assert data_signature(candidate)==EXPECTED_SIGNATURE
        frame=candidate;manifest=path;break
    except Exception as exc:print(path.name,type(exc).__name__,str(exc)[:160],flush=True)
assert frame is not None,'Restore the matching audited manifest. Do not create a new split.'
assert SOURCE_CSV.is_file() and file_hash(SOURCE_CSV)==audit['source_splits_sha256']
fields=['url','ok','label','domain','fold','split']
raw=pd.read_csv(SOURCE_CSV,keep_default_na=False,usecols=['path']+fields)
assert raw.path.is_unique and len(raw)==audit['original_samples']
keys=frame.source_path if 'source_path' in frame else frame.path
assert keys.is_unique
aligned=raw.set_index('path').loc[keys]
for field in fields:
    if field in ['ok','label','fold']:
        assert np.array_equal(pd.to_numeric(frame[field]).to_numpy(),pd.to_numeric(aligned[field]).to_numpy()),field
    else:
        assert np.array_equal(frame[field].fillna('').astype(str).to_numpy(),aligned[field].fillna('').astype(str).to_numpy()),field
assert frame.groupby('sha256').split.nunique().max()==1
assert frame.groupby('sha256').label.nunique().max()==1
assert frame.groupby('domain',dropna=False).split.nunique().max()==1
assert frame[frame.ok.eq(1)].groupby('url').split.nunique().max()==1
test=frame[frame.split.eq('test')].reset_index(drop=True)
assert len(test)==146425 and test.label.value_counts().to_dict()=={1:86120,0:60305}
test['url']=test.url.fillna('').astype(str)
test['available']=test.ok.eq(1)&test.url.str.strip().ne('')
del raw,aligned,frame,candidate;gc.collect()
OUT=RUN/f'final_test_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:6]}'
OUT.mkdir(parents=True,exist_ok=False)
atomic_json(dict(weights=str(WEIGHTS),weights_sha256=WEIGHTS_SHA,selected_epoch=1,
    data_signature=EXPECTED_SIGNATURE,manifest=str(manifest),systems=SYSTEMS,
    threshold=THRESHOLD,max_len=MAX_LEN,precision='FP32',batch_size=BATCH_SIZE,
    url_available_definition='ok==1 and saved URL is nonblank',
    fusion_policy=checkpoint['missing_url_policy'],
    url_source='Saved decoded URLs from original manifest; no fresh QR decoding or URL access',
    text_missing_policy='Legacy saved fallback URL; results reported separately',
    test_history='Test previously evaluated for visual experiments; this is a documented re-evaluation after Validation model selection',
    text_provenance='Legacy BERT checkpoint lacked data_signature; report training-membership provenance explicitly',
    versions=dict(torch=str(torch.__version__),torchvision=str(torchvision.__version__))),OUT/'evaluation_contract.json')
print('Verified Test:',len(test),'URL available:',int(test.available.sum()),'missing:',int((~test.available).sum()),flush=True)


def make_head(size):
    return nn.Sequential(nn.Linear(size,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(.3),
                         nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Linear(128,2))

class MobileNetVisual(nn.Module):
    def __init__(self):
        super().__init__();self.visual_backbone=torchvision.models.mobilenet_v3_small(weights=None)
        self.visual_backbone.classifier=nn.Identity()
        self.visual_proj=nn.Sequential(nn.Linear(576,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(.3))
        self.fusion=make_head(256)
    def features(self,image):return self.visual_proj(self.visual_backbone(image))

bert_config=BertConfig.from_dict(source_identity['bert_config'])
bert_config._attn_implementation='eager'
assert bert_config.hidden_size==128 and bert_config.num_hidden_layers==2 and bert_config.num_attention_heads==2

class TextModel(nn.Module):
    def __init__(self):
        super().__init__();self.text_backbone=BertModel(bert_config,add_pooling_layer=source_identity['has_pooler'])
        self.text_proj=nn.Sequential(nn.Linear(128,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(.3))
        self.fusion=make_head(256)
    def features(self,tokens):return self.text_proj(self.text_backbone(**tokens,return_dict=True).last_hidden_state[:,0])

class ThreeSystems(nn.Module):
    def __init__(self):
        super().__init__();self.visual=MobileNetVisual();self.text=TextModel();self.fusion=make_head(512)
    def forward(self,images,tokens):
        v=self.visual.features(images);t=self.text.features(tokens)
        return self.visual.fusion(v),self.text.fusion(t),self.fusion(torch.cat([v,t],dim=1))

model=ThreeSystems()
model.load_state_dict(state,strict=True)
# Cross-check exported fusion tensors against the separately saved head.
for key,value in checkpoint['fusion_head_state'].items():assert torch.equal(state['fusion.'+key],value),key
model.to(DEVICE).eval()
for parameter in model.parameters():parameter.requires_grad_(False)
print('PASS: all visual/text/fusion tensors loaded strictly. No weight downloads.',flush=True)

TRANSFORM=T.Compose([T.Resize((224,224)),T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])
class JointTest(Dataset):
    def __init__(self,df):self.paths=df.path.to_numpy();self.urls=df.url.to_numpy();self.labels=df.label.to_numpy(dtype=np.int64)
    def __len__(self):return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as image:x=TRANSFORM(image.convert('RGB'))
        return x,self.urls[i],int(self.labels[i])
loader=DataLoader(JointTest(test),batch_size=BATCH_SIZE,shuffle=False,num_workers=WORKERS,
                  pin_memory=True,prefetch_factor=2 if WORKERS else None)

print('[3/4] Evaluate full Test once for all systems...',flush=True)
parts={name:[] for name in SYSTEMS};all_labels=[];seen=0;started=time.time()
with torch.inference_mode():
    bar=tqdm(loader,total=len(loader),desc='Three systems TEST',mininterval=2)
    for batch,(images,urls,labels) in enumerate(bar,1):
        tokens=tokenizer(list(urls),padding='max_length',truncation=True,max_length=MAX_LEN,return_tensors='pt')
        tokens={key:value.to(DEVICE,non_blocking=True) for key,value in tokens.items()}
        with torch.autocast(device_type='cuda',enabled=False):
            visual_logits,text_logits,head_logits=model(images.to(DEVICE,non_blocking=True),tokens)
        visual_p=visual_logits.softmax(1)[:,1].cpu().numpy()
        text_p=text_logits.softmax(1)[:,1].cpu().numpy()
        head_p=head_logits.softmax(1)[:,1].cpu().numpy()
        mask=test.available.iloc[seen:seen+len(labels)].to_numpy(dtype=bool)
        values=dict(visual=visual_p,text=text_p,feature_fusion=np.where(mask,head_p,visual_p),
                    text_with_visual_fallback=np.where(mask,text_p,visual_p))
        for name,p in values.items():assert np.isfinite(p).all();parts[name].append(p)
        all_labels.append(labels.numpy());seen+=len(labels)
        if batch==1 or batch%100==0 or batch==len(loader):
            speed=seen/max(time.time()-started,1e-6)
            print(f'TEST {seen:,}/{len(test):,} ({seen/len(test):.1%}) | {speed:.1f} images/s | ETA {(len(test)-seen)/max(speed,1e-6)/60:.1f} min',flush=True)
y=np.concatenate(all_labels);probabilities={name:np.concatenate(values) for name,values in parts.items()}
assert seen==len(test) and np.array_equal(y,test.label.to_numpy())
available=test.available.to_numpy(dtype=bool)
assert np.array_equal(probabilities['feature_fusion'][~available],probabilities['visual'][~available])
assert np.array_equal(probabilities['text_with_visual_fallback'][available],probabilities['text'][available])
assert np.array_equal(probabilities['text_with_visual_fallback'][~available],probabilities['visual'][~available])


def metrics(y,p):
    if len(y)==0:return dict(samples=0)
    pred=(p>=THRESHOLD).astype(np.int64)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(samples=int(len(y)),accuracy=float(accuracy_score(y,pred)),
        balanced_accuracy=float(balanced_accuracy_score(y,pred)),
        macro_f1=float(f1_score(y,pred,labels=[0,1],average='macro',zero_division=0)),
        malicious_precision=float(precision_score(y,pred,zero_division=0)),
        malicious_recall=float(recall_score(y,pred,zero_division=0)),
        malicious_f1=float(f1_score(y,pred,zero_division=0)),
        roc_auc=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,
        average_precision=float(average_precision_score(y,p)) if len(np.unique(y))==2 else None,
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))

print('[4/4] Save tables, predictions, confusion matrices and ZIP...',flush=True)
rows=[];predictions=test[['path','label','ok','available','domain','sha256']].copy()
for name,p in probabilities.items():
    pred=(p>=THRESHOLD).astype(np.int64)
    predictions[name+'_probability']=p;predictions[name+'_prediction']=pred
    for group,mask in [('all',np.ones(len(y),dtype=bool)),('url_available',available),('url_missing',~available)]:
        rows.append(dict(system=name,group=group,**metrics(y[mask],p[mask])))
        if mask.any():
            report=classification_report(y[mask],pred[mask],labels=[0,1],target_names=['Benign','Malicious'],digits=5,zero_division=0)
            (OUT/f'{name}_{group}_classification_report.txt').write_text(report)
    fig,ax=plt.subplots(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(y,pred,labels=[0,1],display_labels=['Benign','Malicious'],
        cmap='Blues',values_format='d',colorbar=False,ax=ax)
    ax.set_title(name+' — Full Test');fig.tight_layout();fig.savefig(OUT/f'{name}_confusion_matrix.png',dpi=180);plt.close(fig)
summary=pd.DataFrame(rows)
summary.to_csv(OUT/'test_three_systems_comparison.csv',index=False)
summary[summary.group.eq('all')].to_csv(OUT/'test_all_systems_summary.csv',index=False)
atomic_json(rows,OUT/'test_metrics.json')
predictions.to_csv(OUT/'test_predictions.csv.gz',index=False,compression='gzip')

# Paired correctness counts: descriptive; no independent-image significance claim.
paired_rows=[]
for reference in ['visual','text','text_with_visual_fallback']:
    old=(probabilities[reference]>=THRESHOLD)==y
    new=(probabilities['feature_fusion']>=THRESHOLD)==y
    for group,mask in [('all',np.ones(len(y),dtype=bool)),('url_available',available),('url_missing',~available)]:
        paired_rows.append(dict(reference=reference,comparison='feature_fusion',group=group,samples=int(mask.sum()),
            both_correct=int(np.sum(old&new&mask)),both_wrong=int(np.sum(~old&~new&mask)),
            reference_wrong_fusion_correct=int(np.sum(~old&new&mask)),reference_correct_fusion_wrong=int(np.sum(old&~new&mask))))
pd.DataFrame(paired_rows).to_csv(OUT/'paired_correctness_counts.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for name,p in probabilities.items():
    fpr,tpr,_=roc_curve(y,p);precision,recall,_=precision_recall_curve(y,p)
    axes[0].plot(fpr,tpr,label=name);axes[1].plot(recall,precision,label=name)
axes[0].plot([0,1],[0,1],'--',color='gray')
axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='Full Test ROC')
axes[1].set(xlabel='Recall',ylabel='Precision',title='Full Test precision–recall')
for ax in axes:ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'test_roc_pr.png',dpi=180);plt.close(fig)
assert file_hash(WEIGHTS)==WEIGHTS_SHA,'Source checkpoint changed during evaluation.'
output_files=[p for p in OUT.iterdir() if p.is_file()]
backup_manifest=[dict(file=p.name,bytes=p.stat().st_size,sha256=file_hash(p)) for p in output_files]
atomic_json(dict(status='completed',files=backup_manifest,checkpoint_unchanged=True,test_samples=len(test),
                 note='Fixed-weight evaluation; no training, threshold search, Test-based model selection or fresh QR decoding.'),OUT/'completion_record.json')
zip_path=OUT.with_suffix('.zip');tmp=zip_path.with_suffix('.writing')
with zipfile.ZipFile(tmp,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
    for p in OUT.iterdir():
        if p.is_file():archive.write(p,arcname=OUT.name+'/'+p.name)
with zipfile.ZipFile(tmp) as archive:
    assert archive.testzip() is None
    for entry in backup_manifest:
        h=hashlib.sha256()
        with archive.open(OUT.name+'/'+entry['file']) as f:
            for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
        assert h.hexdigest()==entry['sha256']
os.replace(tmp,zip_path)
shown=summary[summary.group.eq('all')][['system','samples','accuracy','macro_f1','malicious_precision','malicious_recall','malicious_f1','roc_auc']].copy()
for col in ['accuracy','macro_f1','malicious_precision','malicious_recall','malicious_f1']:shown[col]*=100
print('\n========== FINAL FULL TEST RESULTS ==========')
print(shown.round(5).to_string(index=False))
print('\nSubgroups (Accuracy/Macro-F1 as fractions):')
print(summary[['system','group','samples','accuracy','macro_f1','roc_auc']].to_string(index=False))
print('Results:',OUT,flush=True)
print('Verified results ZIP:',zip_path,flush=True)
print('Weights unchanged. Test was used for reporting only.',flush=True)
